# BlockChain and CryptoCurrency

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
import hashlib
import json
import time
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

class Block:
    def __init__(self, index, transactions, timestamp, previous_hash, nonce=0):
        self.index = index
        self.transactions = transactions
        self.timestamp = timestamp
        self.previous_hash = previous_hash
        self.nonce = nonce
        self.hash = self.calculate_hash()

    def calculate_hash(self):
        block_string = json.dumps({
            "index": self.index,
            "transactions": self.transactions,
            "timestamp": self.timestamp,
            "previous_hash": self.previous_hash,
            "nonce": self.nonce
        }, sort_keys=True)
        return hashlib.sha256(block_string.encode()).hexdigest()

    def mine_block(self, difficulty):
        target = "0" * difficulty
        start_time = time.time()

        while self.hash[:difficulty] != target:
            self.nonce += 1
            self.hash = self.calculate_hash()

        mining_time = time.time() - start_time
        print(f"Block mined: {self.hash[:20]}... (Nonce: {self.nonce}, Time: {mining_time:.2f}s)")
        return mining_time

class Transaction:
    def __init__(self, from_address, to_address, amount, fee=0):
        self.from_address = from_address
        self.to_address = to_address
        self.amount = amount
        self.fee = fee
        self.timestamp = time.time()
        self.signature = self.calculate_signature()

    def calculate_signature(self):
        tx_string = f"{self.from_address}{self.to_address}{self.amount}{self.timestamp}"
        return hashlib.sha256(tx_string.encode()).hexdigest()[:16]

    def to_dict(self):
        return {
            "from": self.from_address,
            "to": self.to_address,
            "amount": self.amount,
            "fee": self.fee,
            "timestamp": self.timestamp,
            "signature": self.signature
        }

class Blockchain:
    def __init__(self):
        self.chain = [self.create_genesis_block()]
        self.difficulty = 2
        self.pending_transactions = []
        self.mining_reward = 100
        self.balances = {}
        self.mining_stats = []

    def create_genesis_block(self):
        genesis_transactions = [{"from": "genesis", "to": "system", "amount": 1000000, "fee": 0}]
        return Block(0, genesis_transactions, time.time(), "0")

    def get_latest_block(self):
        return self.chain[-1]

    def add_transaction(self, transaction):
        if self.get_balance(transaction.from_address) >= transaction.amount + transaction.fee:
            self.pending_transactions.append(transaction.to_dict())
            return True
        return False

    def mine_pending_transactions(self, mining_reward_address):
        reward_transaction = {
            "from": "system",
            "to": mining_reward_address,
            "amount": self.mining_reward,
            "fee": 0
        }

        self.pending_transactions.append(reward_transaction)

        block = Block(
            len(self.chain),
            self.pending_transactions,
            time.time(),
            self.get_latest_block().hash
        )

        mining_time = block.mine_block(self.difficulty)
        self.mining_stats.append({
            'block_number': len(self.chain),
            'mining_time': mining_time,
            'transactions': len(self.pending_transactions),
            'difficulty': self.difficulty,
            'timestamp': datetime.now()
        })

        self.chain.append(block)
        self.pending_transactions = []
        self.update_balances()

        return block

    def update_balances(self):
        self.balances = {}

        for block in self.chain:
            for tx in block.transactions:
                if tx["from"] != "genesis" and tx["from"] != "system":
                    self.balances[tx["from"]] = self.balances.get(tx["from"], 0) - tx["amount"] - tx["fee"]

                if tx["to"] != "system":
                    self.balances[tx["to"]] = self.balances.get(tx["to"], 0) + tx["amount"]

    def get_balance(self, address):
        return self.balances.get(address, 0)

    def is_chain_valid(self):
        for i in range(1, len(self.chain)):
            current_block = self.chain[i]
            previous_block = self.chain[i-1]

            if current_block.hash != current_block.calculate_hash():
                return False

            if current_block.previous_hash != previous_block.hash:
                return False

        return True

    def get_transaction_history(self, address):
        history = []
        for block in self.chain:
            for tx in block.transactions:
                if tx["from"] == address or tx["to"] == address:
                    history.append({
                        'block': block.index,
                        'transaction': tx,
                        'timestamp': block.timestamp
                    })
        return history

class CryptocurrencySimulator:
    def __init__(self):
        self.blockchain = Blockchain()
        self.users = {}
        self.market_data = []
        self.base_price = 100
        self.current_price = self.base_price

    def create_user(self, username, initial_balance=1000):
        self.users[username] = {
            'balance': initial_balance,
            'address': f"addr_{username}_{hash(username) % 10000}"
        }
        self.blockchain.balances[self.users[username]['address']] = initial_balance
        return self.users[username]['address']

    def transfer(self, from_user, to_user, amount, fee=1):
        if from_user not in self.users or to_user not in self.users:
            return False

        from_addr = self.users[from_user]['address']
        to_addr = self.users[to_user]['address']

        tx = Transaction(from_addr, to_addr, amount, fee)
        success = self.blockchain.add_transaction(tx)

        if success:
            print(f"Transaction added: {from_user} -> {to_user}: {amount} coins")
        else:
            print(f"Transaction failed: Insufficient balance")

        return success

    def mine_block(self, miner_user):
        if miner_user not in self.users:
            return None

        miner_addr = self.users[miner_user]['address']
        block = self.blockchain.mine_pending_transactions(miner_addr)

        self.update_market_price()

        return block

    def update_market_price(self):
        total_supply = sum(self.blockchain.balances.values())
        transaction_volume = len(self.blockchain.pending_transactions)

        price_change = np.random.normal(0, 5) + (transaction_volume * 0.5)
        self.current_price = max(1, self.current_price + price_change)

        self.market_data.append({
            'timestamp': datetime.now(),
            'price': self.current_price,
            'volume': transaction_volume,
            'supply': total_supply,
            'blocks': len(self.blockchain.chain)
        })

    def get_portfolio(self, username):
        if username not in self.users:
            return None

        addr = self.users[username]['address']
        balance = self.blockchain.get_balance(addr)
        value = balance * self.current_price

        return {
            'address': addr,
            'balance': balance,
            'value_usd': value,
            'price_per_coin': self.current_price
        }

    def simulate_trading_day(self):
        users = list(self.users.keys())

        for _ in range(10):
            if len(users) >= 2:
                from_user = np.random.choice(users)
                to_user = np.random.choice([u for u in users if u != from_user])
                amount = np.random.randint(1, 50)

                self.transfer(from_user, to_user, amount)

            if np.random.random() < 0.3:
                miner = np.random.choice(users)
                self.mine_block(miner)

    def generate_analytics(self):
        print("=== BLOCKCHAIN ANALYTICS ===\n")

        print(f"Total Blocks: {len(self.blockchain.chain)}")
        print(f"Chain Valid: {self.blockchain.is_chain_valid()}")
        print(f"Current Difficulty: {self.blockchain.difficulty}")
        print(f"Pending Transactions: {len(self.blockchain.pending_transactions)}")
        print(f"Current Price: ${self.current_price:.2f}")

        print("\n=== USER PORTFOLIOS ===")
        for username in self.users:
            portfolio = self.get_portfolio(username)
            print(f"{username}: {portfolio['balance']:.2f} coins (${portfolio['value_usd']:.2f})")

        if self.blockchain.mining_stats:
            avg_mining_time = np.mean([stat['mining_time'] for stat in self.blockchain.mining_stats])
            print(f"\nAverage Mining Time: {avg_mining_time:.2f} seconds")

        self.visualize_blockchain()

    def visualize_blockchain(self):
        if len(self.market_data) < 2:
            return

        df = pd.DataFrame(self.market_data)

        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))

        ax1.plot(df['timestamp'], df['price'], color='blue', linewidth=2)
        ax1.set_title('Cryptocurrency Price Over Time')
        ax1.set_ylabel('Price ($)')
        ax1.tick_params(axis='x', rotation=45)

        ax2.bar(range(len(self.blockchain.chain)),
                [len(block.transactions) for block in self.blockchain.chain])
        ax2.set_title('Transactions per Block')
        ax2.set_xlabel('Block Number')
        ax2.set_ylabel('Transaction Count')

        if self.blockchain.mining_stats:
            mining_df = pd.DataFrame(self.blockchain.mining_stats)
            ax3.plot(mining_df['block_number'], mining_df['mining_time'],
                    marker='o', color='red')
            ax3.set_title('Mining Time per Block')
            ax3.set_xlabel('Block Number')
            ax3.set_ylabel('Mining Time (seconds)')

        balances = list(self.blockchain.balances.values())
        ax4.hist(balances, bins=10, alpha=0.7, color='green')
        ax4.set_title('Balance Distribution')
        ax4.set_xlabel('Balance')
        ax4.set_ylabel('Frequency')

        plt.tight_layout()
        plt.show()

    def display_blockchain_structure(self):
        print("\n=== BLOCKCHAIN STRUCTURE ===")
        for i, block in enumerate(self.blockchain.chain):
            print(f"\nBlock {i}:")
            print(f"  Hash: {block.hash[:20]}...")
            print(f"  Previous Hash: {block.previous_hash[:20]}...")
            print(f"  Nonce: {block.nonce}")
            print(f"  Transactions: {len(block.transactions)}")

            if i > 0:
                for tx in block.transactions[:3]:
                    print(f"    {tx['from'][:10]}... -> {tx['to'][:10]}...: {tx['amount']}")

def main():
    print("=== BLOCKCHAIN & CRYPTOCURRENCY SIMULATOR ===\n")

    simulator = CryptocurrencySimulator()

    users = ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve']
    for user in users:
        simulator.create_user(user, initial_balance=1000)

    print("Created users with initial balances\n")

    simulator.simulate_trading_day()

    simulator.generate_analytics()
    simulator.display_blockchain_structure()

    print("\n=== TRANSACTION HISTORY SAMPLE ===")
    alice_addr = simulator.users['Alice']['address']
    history = simulator.blockchain.get_transaction_history(alice_addr)

    for tx_record in history[:5]:
        tx = tx_record['transaction']
        print(f"Block {tx_record['block']}: {tx['from'][:10]}... -> {tx['to'][:10]}...: {tx['amount']}")

if __name__ == "__main__":
    main()